# NEXUS Local AI — Transport Edition

**Dean A. Kulik · QuHarmonics Research Group**  
ORCID 0009-0003-3128-8828

Single Python code cell. Load a model in LM Studio, start its local server on port **1235**, then run the cell.

Commands: `:state`, `:step`, `:reverse`, `:transport 5`, `:clear`, `:quit`. The model can also request the corresponding tools in plain language. The notebook reuses `NEXUS_local_AI_events.jsonl` in the notebook's current working directory.

Transport is **XOR of word locations**, not a claim about native ring **array indices**. The compiler checks `F H_h = H_h F`, inverse commute and exact state return over randomized populated fields. The event log extends older records with a SHA-256 linked chain; a separate trusted digest anchor would be required to detect full-history replacement.


In [ ]:
"""NEXUS local AI / compiler bridge — transport edition.
Dean A. Kulik, QuHarmonics Research Group, ORCID 0009-0003-3128-8828.

Python 3.10+, standard library only. Runs as a script or in one Jupyter cell.
LM Studio: http://localhost:1235/v1 (override with LM_STUDIO_URL).

Machine state: (p, w, D) over n-bit binary *word locations* (not native ring indices).
F: x=D[p]^w; p'=p^x; w'=P_n(x); D'[p]=D[p]^w'.
H_h: p'=p^h; w'=w; D_h[q]=D[q^h].
F H_h = H_h F, H_h^(-1)=H_h, H_a H_b=H_(a^b).

The local language model requests tools; the compiler computes and verifies operations.
No unrestricted shell or filesystem tool is exposed to the model.
"""
from __future__ import annotations
import hashlib
import json
import os
import random
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

BASE = os.getenv('LM_STUDIO_URL', 'http://localhost:1235/v1').rstrip('/')
if BASE.endswith('/v1'):
    BASE = BASE[:-3]
TOKEN = os.getenv('LM_API_TOKEN', '')
WIDTH, SEED_P, SEED_W = 4, 0, 1
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
LEDGER = ROOT / 'NEXUS_local_AI_events.jsonl'  # Reuse & verify the original history.
GENESIS = '0' * 64


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)


def chain_hash(previous, payload):
    return hashlib.sha256((previous + '\n' + canonical(payload)).encode('utf-8')).hexdigest()


def partner(x: int, n: int) -> int:
    """Native P: reverse nonpalindromes, complement palindromes."""
    mask = (1 << n) - 1
    if not (2 <= n <= 16 and 0 <= x <= mask):
        raise ValueError('n must be 2..16 and x must fit n bits')
    rev = int(f'{x:0{n}b}'[::-1], 2)
    return rev if rev != x else x ^ mask


class Machine:
    def __init__(self, n=WIDTH, p=SEED_P, w=SEED_W, field=None):
        if not (2 <= n <= 16):
            raise ValueError('n must be 2..16 for this prototype')
        mask = (1 << n) - 1
        if not (0 <= p <= mask and 0 <= w <= mask):
            raise ValueError('p and w must be n-bit words')
        self.n, self.p, self.w = n, p, w
        self.D = {int(k): int(v) for k, v in (field or {}).items() if int(v)}
        if any(not (0 <= k <= mask and 0 <= v <= mask) for k, v in self.D.items()):
            raise ValueError('field keys and values must be n-bit words')
        self.last_hash = GENESIS

    def copy(self):
        other = Machine(self.n, self.p, self.w, self.D)
        other.last_hash = self.last_hash
        return other

    def adopt(self, other):
        self.p, self.w, self.D = other.p, other.w, other.D.copy()
        self.last_hash = other.last_hash

    def snapshot(self):
        return {'n': self.n, 'p': self.p, 'w': self.w,
                'field': {str(k): v for k, v in sorted(self.D.items())}}

    def _write(self, at, value):
        if value:
            self.D[at] = value
        else:
            self.D.pop(at, None)

    def forward(self):
        oldp = self.p
        x = self.D.get(oldp, 0) ^ self.w
        w2 = partner(x, self.n)
        self._write(oldp, self.D.get(oldp, 0) ^ w2)
        self.p, self.w = oldp ^ x, w2
        return {'direction': 'forward', 'x': x, 'written_at': oldp}

    def backward(self):
        x = partner(self.w, self.n)
        oldp = self.p ^ x
        oldw = self.w
        recovered = self.D.get(oldp, 0) ^ oldw
        self._write(oldp, recovered)
        self.p, self.w = oldp, x ^ recovered
        return {'direction': 'reverse', 'x': x, 'restored_at': oldp}

    def transport(self, h: int):
        if type(h) is not int or not 0 <= h < (1 << self.n):
            raise ValueError(f'transport h must be an integer 0..{(1 << self.n)-1}')
        prev = self.p
        self.p ^= h
        self.D = {k ^ h: v for k, v in self.D.items()}
        return {'direction': 'transport', 'h': h, 'from_word': prev,
                'to_word': self.p, 'relocated_field_cells': len(self.D)}


def verify():
    """Executed deterministic checks: involution, random F/H commuting squares, routes."""
    checks = 0
    for n in range(2, 11):
        for x in range(1 << n):
            assert partner(partner(x, n), n) == x
            assert partner(x, n) != x
            checks += 1
    rng = random.Random(181)
    for n in range(2, 17):
        mask = (1 << n) - 1
        for _ in range(320):
            D = {}
            for __ in range(1 + rng.randrange(12)):
                k = rng.randrange(mask + 1)
                v = rng.randrange(mask + 1)
                if v:
                    D[k] = v
            m = Machine(n, rng.randrange(mask+1), rng.randrange(mask+1), D)
            h, a, b = (rng.randrange(mask+1) for __ in range(3))
            # H is its own inverse; transport group is XOR.
            t = m.copy(); t.transport(h); t.transport(h)
            assert t.snapshot() == m.snapshot()
            t = m.copy(); t.transport(a); t.transport(b)
            u = m.copy(); u.transport(a ^ b)
            assert t.snapshot() == u.snapshot()
            # The commuting square F o H == H o F.
            left = m.copy(); left.transport(h); left.forward()
            right = m.copy(); right.forward(); right.transport(h)
            assert left.snapshot() == right.snapshot(), ('FH=HF', n)
            # Also commute for inverse steps.
            left = m.copy(); left.transport(h); left.backward()
            right = m.copy(); right.backward(); right.transport(h)
            assert left.snapshot() == right.snapshot(), ('F^-1H=HF^-1', n)
            # Roundtrip on arbitrary populated field, not just vacuum seed.
            t = m.copy(); t.forward(); t.backward()
            assert t.snapshot() == m.snapshot()
            t = m.copy(); t.backward(); t.forward()
            assert t.snapshot() == m.snapshot()
            checks += 6
    for n in (4, 8, 16):
        m = Machine(n=n, p=0, w=1)
        initial = m.snapshot()
        for _ in range(1000):
            m.forward()
        for _ in range(1000):
            m.backward()
        assert m.snapshot() == initial
    print(f'RUNS: {checks:,} deterministic conditions; P, arbitrary-field roundtrip, '
          'transport involution/group law, F H = H F, inverse commutation, '
          'and 1000-step return at n=4,8,16.')


def apply_mutation(machine, kind, h=None):
    if kind == 'step':
        return machine.forward()
    if kind == 'reverse':
        return machine.backward()
    if kind == 'transport':
        return machine.transport(h)
    raise ValueError('unknown operation')


def replay():
    m = Machine()
    if not LEDGER.exists():
        return m
    count, legacy = 0, 0
    with LEDGER.open('r', encoding='utf-8') as f:
        for count, line in enumerate(f, 1):
            event = json.loads(line)
            if event['before'] != m.snapshot():
                raise RuntimeError(f'Ledger discontinuity before line {count}')
            kind = event['operation']
            # Supports old unhashed events without rewriting or discarding them.
            if 'event_hash' in event:
                if event.get('prev_hash') != m.last_hash:
                    raise RuntimeError(f'Ledger hash link invalid at line {count}')
                payload = {k: v for k, v in event.items()
                           if k not in ('prev_hash', 'event_hash')}
                digest = chain_hash(m.last_hash, payload)
                if digest != event['event_hash']:
                    raise RuntimeError(f'Ledger event hash invalid at line {count}')
            else:
                legacy += 1
                payload = event
                digest = chain_hash(m.last_hash, payload)
            meta = apply_mutation(m, kind, event.get('route', {}).get('h'))
            if event['after'] != m.snapshot() or (event.get('route') and meta != event['route']):
                raise RuntimeError(f'Ledger event mismatch after line {count}')
            m.last_hash = digest
    print(f'Replayed {count} event(s), including {legacy} legacy event(s). '
          f'Last chain digest: {m.last_hash}')
    return m


def operation(m: Machine, kind: str, h=None):
    if kind == 'state':
        return {'status': 'RUNS', 'state': m.snapshot(), 'ledger_head': m.last_hash}
    if kind not in ('step', 'reverse', 'transport'):
        return {'error': 'Only state, step, reverse, transport are allowed'}
    before = m.snapshot()
    trial = m.copy()
    try:
        meta = apply_mutation(trial, kind, h)
    except ValueError as e:
        return {'error': str(e), 'status': 'OPEN'}
    after = trial.snapshot()
    # Independent compiler return; never ask the model to certify it.
    check = trial.copy()
    if kind == 'step':
        check.backward()
    elif kind == 'reverse':
        check.forward()
    else:
        check.transport(h)
    assert check.snapshot() == before, f'{kind}: way home failed'
    payload = {'version': 2, 'operation': kind, 'before': before,
               'after': after, 'route': meta}
    digest = chain_hash(m.last_hash, payload)
    record = {**payload, 'prev_hash': m.last_hash, 'event_hash': digest}
    # Write before committing the in-memory state, so a disk failure cannot create
    # an unrecorded accepted transition. Existing ledger lines are never rewritten.
    with LEDGER.open('a', encoding='utf-8') as f:
        f.write(canonical(record) + '\n')
        f.flush()
        os.fsync(f.fileno())
    trial.last_hash = digest
    m.adopt(trial)
    return {'status': 'RUNS', **record, 'return_verified': True}


def request_json(method, endpoint, payload=None, timeout=180):
    headers = {'Content-Type': 'application/json'}
    if TOKEN:
        headers['Authorization'] = f'Bearer {TOKEN}'
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    req = Request(BASE + endpoint, headers=headers, data=data, method=method)
    with urlopen(req, timeout=timeout) as res:
        return json.loads(res.read().decode('utf-8'))


def find_model():
    data = request_json('GET', '/api/v1/models')
    loaded = [(model['display_name'], inst['id'])
              for model in data.get('models', []) if model.get('type') == 'llm'
              for inst in model.get('loaded_instances', [])]
    if not loaded:
        print('No loaded LLM. Load Gemma 4 E4B in LM Studio, then rerun.')
        return None
    for i, (label, ident) in enumerate(loaded, 1):
        print(f'  {i}. {label} [{ident}]')
    print('Using first loaded model:', loaded[0][0])
    return loaded[0][1]


TOOLS = [
    {'type':'function','function':{'name':'nexus_state',
       'description':'Read current exact reader, carry, sparse field, ledger digest.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_step',
       'description':'Execute one exact reversible native partner reader-field step.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_reverse',
       'description':'Execute exact inverse native partner reader-field step.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_transport',
       'description':'Relocate full reader-field state by n-bit XOR offset h. '
         'Preserves behavior: F H_h = H_h F and H_h is its own inverse.',
       'parameters':{'type':'object','properties':{'h':{
           'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1,
           'description':'XOR word offset, integer 0..15 for current n=4'}},
           'required':['h'],'additionalProperties':False}}},
]

SYSTEM = ('You interface with an independently executing NEXUS binary compiler. '
          'Use tools for actual machine state or changes. Never invent tool results. '
          'Transport h is an integer XOR word offset; it moves the reader and all '
          'field keys while keeping carried state and field values. '
          'State clearly which operations actually ran. The compiler owns verification.')


def ask_model(model_id, machine, user_text, history=None):
    # History is optional; if supplied, context persists within this Python session.
    messages = history if history is not None else [{'role':'system','content':SYSTEM}]
    messages.append({'role':'user','content':user_text})
    for _ in range(16):
        response = request_json('POST', '/v1/chat/completions', {
            'model': model_id, 'messages': messages, 'tools': TOOLS,
            'tool_choice':'auto', 'temperature':0.1, 'max_tokens':640,
            'stream':False,
        })
        msg = response['choices'][0]['message']
        calls = msg.get('tool_calls') or []
        if not calls:
            content = msg.get('content') or '(no text returned)'
            messages.append({'role':'assistant','content':content})
            print('\nMODEL:', content)
            return
        messages.append({'role':'assistant','content':msg.get('content'),
                         'tool_calls':calls})
        for call in calls:
            fn = call.get('function',{}).get('name','')
            kind = {'nexus_state':'state','nexus_step':'step',
                    'nexus_reverse':'reverse','nexus_transport':'transport'}.get(fn,'')
            try:
                args = json.loads(call.get('function',{}).get('arguments') or '{}')
                if not isinstance(args,dict):
                    raise ValueError('arguments must be a JSON object')
                if kind == 'transport' and 'h' not in args:
                    raise ValueError('nexus_transport requires h')
                result = operation(machine, kind, args.get('h'))
            except (ValueError, TypeError, json.JSONDecodeError) as err:
                result = {'status':'OPEN','error':str(err)}
            print(f'  TOOL {fn}:', canonical(result))
            messages.append({'role':'tool','tool_call_id':call['id'],
                             'content':canonical(result)})
    print('Model tool-round budget reached; exact machine state:', canonical(machine.snapshot()))


def main():
    verify()
    machine = replay()
    print('Current state:', canonical(machine.snapshot()))
    print('LM Studio:', BASE)
    try:
        model_id = find_model()
    except (URLError, HTTPError, TimeoutError, ValueError) as err:
        print('LM Studio not reachable:', err)
        model_id = None
    chat = [{'role':'system','content':SYSTEM}]
    print('Commands: :state, :step, :reverse, :transport <h>, :clear, :quit')
    while True:
        try:
            line = input('nexus> ').strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not line: continue
        if line == ':quit': break
        if line == ':clear':
            chat[:] = [{'role':'system','content':SYSTEM}]
            print('Conversation context reset; deterministic machine unchanged.')
        elif line in (':state',':step',':reverse'):
            print(json.dumps(operation(machine,line[1:]),indent=2))
        elif line.startswith(':transport'):
            parts = line.split()
            if len(parts) != 2:
                print('Usage: :transport 3  (also accepts 0x3)')
                continue
            try:
                h = int(parts[1],0)
                print(json.dumps(operation(machine,'transport',h),indent=2))
            except ValueError as err:
                print('Invalid transport:', err)
        elif not model_id:
            print('Model unavailable; compiler commands still work.')
        else:
            try:
                ask_model(model_id,machine,line,chat)
            except (URLError,HTTPError,TimeoutError,KeyError,ValueError) as err:
                print('LM Studio API error:',err)


if __name__ == '__main__':
    main()


RUNS: 30,844 deterministic conditions; P, arbitrary-field roundtrip, transport involution/group law, F H = H F, inverse commutation, and 1000-step return at n=4,8,16.
Replayed 4 event(s), including 4 legacy event(s). Last chain digest: 766184a20229f53fa6422979c87ce9ee861ee50a1db9d46783f36ac050f15d09
Current state: {"field":{},"n":4,"p":0,"w":1}
LM Studio: http://localhost:1235
  1. Gemma 4 E4B [google/gemma-4-e4b]
Using first loaded model: Gemma 4 E4B
Commands: :state, :step, :reverse, :transport <h>, :clear, :quit


nexus>  :state :transport 5 :step :reverse :transport 5 :state


  TOOL nexus_state: {"ledger_head":"766184a20229f53fa6422979c87ce9ee861ee50a1db9d46783f36ac050f15d09","state":{"field":{},"n":4,"p":0,"w":1},"status":"RUNS"}
  TOOL nexus_transport: {"after":{"field":{},"n":4,"p":5,"w":1},"before":{"field":{},"n":4,"p":0,"w":1},"event_hash":"49e31319f3c6efcdb271dab12213a31e5252040fc6157f3309328f0cae006bca","operation":"transport","prev_hash":"766184a20229f53fa6422979c87ce9ee861ee50a1db9d46783f36ac050f15d09","return_verified":true,"route":{"direction":"transport","from_word":0,"h":5,"relocated_field_cells":0,"to_word":5},"status":"RUNS","version":2}
  TOOL nexus_step: {"after":{"field":{"5":8},"n":4,"p":4,"w":8},"before":{"field":{},"n":4,"p":5,"w":1},"event_hash":"280cb4d0fce9e0af0933902edee2f3bdf5b7a03a899692f237d814b91c3c2714","operation":"step","prev_hash":"49e31319f3c6efcdb271dab12213a31e5252040fc6157f3309328f0cae006bca","return_verified":true,"route":{"direction":"forward","written_at":5,"x":1},"status":"RUNS","version":2}
  TOOL nexus_reverse: {"

nexus>  Use the Nexus tools to read the current state, transport the complete reader-field system by XOR offset 5, execute one forward step, reverse it, and transport by 5 again. Verify that the final state matches the initial state. Report the actual tool results.


  TOOL nexus_state: {"ledger_head":"fad65933ed8657bc9ac6a3aad0ffff70c5c67c8bc31347f1b2a39a8dfd7e5440","state":{"field":{},"n":4,"p":0,"w":1},"status":"RUNS"}
  TOOL nexus_transport: {"after":{"field":{},"n":4,"p":5,"w":1},"before":{"field":{},"n":4,"p":0,"w":1},"event_hash":"f4250f27e2f0c85d2b54559726bc8aebf07f5e6add5192137fe52af27c2b0d05","operation":"transport","prev_hash":"fad65933ed8657bc9ac6a3aad0ffff70c5c67c8bc31347f1b2a39a8dfd7e5440","return_verified":true,"route":{"direction":"transport","from_word":0,"h":5,"relocated_field_cells":0,"to_word":5},"status":"RUNS","version":2}
  TOOL nexus_step: {"after":{"field":{"5":8},"n":4,"p":4,"w":8},"before":{"field":{},"n":4,"p":5,"w":1},"event_hash":"3099630ba95f94bcef8b02b1e8227a43ccdcba042d8e8e7a229b02ea47a2897f","operation":"step","prev_hash":"f4250f27e2f0c85d2b54559726bc8aebf07f5e6add5192137fe52af27c2b0d05","return_verified":true,"route":{"direction":"forward","written_at":5,"x":1},"status":"RUNS","version":2}
  TOOL nexus_reverse: {"

nexus>  :step :step :transport 5 :transport 5 :reverse :transport 5 :step


  TOOL nexus_step: {"after":{"field":{"0":8},"n":4,"p":1,"w":8},"before":{"field":{},"n":4,"p":0,"w":1},"event_hash":"ed4dd3cb621b8b02553f3fcc26fbe1cda65bd1721e9efb68a79a9c02854f57e6","operation":"step","prev_hash":"540606f21f5941ff336b91ad4e343ccb9ed98fe1fb4356a9f94ce150f1c8b52a","return_verified":true,"route":{"direction":"forward","written_at":0,"x":1},"status":"RUNS","version":2}
  TOOL nexus_step: {"after":{"field":{"0":8,"1":1},"n":4,"p":9,"w":1},"before":{"field":{"0":8},"n":4,"p":1,"w":8},"event_hash":"ec88a2d7f6938433a92cb8a2c6c9c91fcf9703c69251f90e8f3c6c7097dfc1a4","operation":"step","prev_hash":"ed4dd3cb621b8b02553f3fcc26fbe1cda65bd1721e9efb68a79a9c02854f57e6","return_verified":true,"route":{"direction":"forward","written_at":1,"x":8},"status":"RUNS","version":2}
  TOOL nexus_transport: {"after":{"field":{"4":1,"5":8},"n":4,"p":12,"w":1},"before":{"field":{"0":8,"1":1},"n":4,"p":9,"w":1},"event_hash":"6550243da48acf162a31bd6bf79fc3d1251e243ff91626f150d61adb6bacbbb6","operati

nexus>  :step


{
  "status": "RUNS",
  "version": 2,
  "operation": "step",
  "before": {
    "n": 4,
    "p": 4,
    "w": 8,
    "field": {
      "5": 8
    }
  },
  "after": {
    "n": 4,
    "p": 12,
    "w": 1,
    "field": {
      "4": 1,
      "5": 8
    }
  },
  "route": {
    "direction": "forward",
    "x": 8,
    "written_at": 4
  },
  "prev_hash": "0345300c336db0e569bec12760154e6f5674e6f3f858ec4a84437ab1e80dae87",
  "event_hash": "d3c28760a59dc2aa6f4de437de31d217e0e9e976af823b3c183a047cb1513077",
  "return_verified": true
}
